<a href="https://colab.research.google.com/github/narpavi-ai/lasp-agent-workshop/blob/main/notebooks/01-plain-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 · Ask the model

**LASP workshop: build your agent**

You have pure-component CO₂ and N₂ isotherms for a 13X-like adsorbent.
You want one number: **the CO₂/N₂ selectivity for flue gas (15% CO₂) at
1 bar.** In the lab you'd fit both isotherms, then run IAST.

Here, you ask a language model, and find out exactly what it can and
can't do. About **5 requests**.

## Before you start

Add these with the **🔑 key icon** in Colab's left sidebar, and turn **Notebook access** on for each. Notebook access is per notebook, so each lab asks again.

| Name | Value |
|---|---|
| `GOOGLE_API_KEY` | your free key from <https://aistudio.google.com/apikey> |
| `ANTHROPIC_API_KEY` | *optional:* only if you switch to Claude (a paid key from <https://console.anthropic.com>) |

## Setup

Run these in order. Each takes a few seconds.

### Setup · Install

LangChain with its Gemini and Claude connectors, at the exact versions we tested, plus this workshop's helper file.

In [ ]:
# pip may print a warning about google-auth and google-colab. That's expected
# and harmless: Gemini's library needs a newer google-auth than Colab ships.
%pip install -q langchain==1.4.7 langchain-core==1.7.0 langchain-google-genai==4.4.2 langchain-anthropic==1.7.6
!curl -sSfO https://raw.githubusercontent.com/narpavi-ai/lasp-agent-workshop/main/notebooks/lasp_lab.py
print("✅ Installed, and helpers downloaded.")

### Setup · Helpers

Display and connection helpers, so the notebook shows only what's being taught. Curious? Read [`lasp_lab.py`](lasp_lab.py).

In [ ]:
from lasp_lab import *   # show(), call(), run_live() and friends: see lasp_lab.py
print("✅ Helpers ready.")

### Setup · Choose your model

`"gemini"` is free and needs `GOOGLE_API_KEY`. `"claude"` is paid and needs `ANTHROPIC_API_KEY`. Nothing else in the lab changes.

In [ ]:
PROVIDER = "gemini"    # "gemini" or "claude"

### Setup · Load your key

Read from Colab Secrets, so it never appears in the notebook.

In [ ]:
import os
KEY_NAME = {"gemini": "GOOGLE_API_KEY", "claude": "ANTHROPIC_API_KEY"}[PROVIDER]
os.environ[KEY_NAME] = secret(KEY_NAME)
print(f"✅ {KEY_NAME} loaded.")

### Setup · Create the model

One line picks the provider and model. Everything after it is the same for both.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

MODELS = {"gemini": "google_genai:gemini-3.5-flash-lite", "claude": "anthropic:claude-haiku-5-5"}
MODEL = MODELS[PROVIDER]

# Gemini's free tier allows 15 requests a minute; stay under it. Paid Claude keys allow more.
limiter = InMemoryRateLimiter(requests_per_second=0.2) if PROVIDER == "gemini" else None

# Claude 5 thinks before answering. Low effort keeps that short (faster, cheaper),
# and the cap leaves room for the answer after the thinking.
extra = {"max_tokens": 16000, "thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}} if PROVIDER == "claude" else {}
model = init_chat_model(MODEL, rate_limiter=limiter, **extra)
print(f"✅ Model ready: {MODEL}")

### Step 1 · One call: a system prompt and a user prompt

Every call sends a list of **messages**, each with a role:

| Role | Who writes it | What it's for |
|---|---|---|
| `system` | you, the developer | standing instructions: who the model is, what rules it follows |
| `user` | the person asking | the question |

What comes back is a response object, not just text.

In [ ]:
SYSTEM_PROMPT = ("You are a research assistant at the Laboratory for Advanced Separation Processes (LASP), "
                 "led by Prof. Arvind Rajendran at the University of Alberta. The group works on "
                 "adsorption-based separations such as CO2 capture.")
USER_PROMPT = "In two sentences: what does Ideal Adsorbed Solution Theory assume?"

response = model.invoke([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
])
show(response, "Model answer", "A well-documented concept, so this is where a language model is strong.")
print(response.usage_metadata)   # what this call cost, in tokens

### Step 2 · The real question

Same model and system prompt. Now the user prompt carries **your data**,
and the answer depends on it.

In [ ]:
# Synthetic 13X-like data made for this workshop (not measured), 298 K.
# The same numbers are on the server as workshop_co2_298K_synthetic and workshop_n2_298K_synthetic.
pressure_bar = [0.001, 0.003, 0.01, 0.03, 0.1, 0.2, 0.4, 0.6, 0.8, 1, 1.5, 2, 3, 5, 7, 10]
co2_mol_per_kg = [0.147171, 0.399272, 1.02996, 1.9223, 2.88179, 3.25169, 3.86714, 4.16431, 4.24648, 4.41152, 4.66168, 4.89841, 5.10685, 5.2541, 5.17341, 5.23311]
n2_mol_per_kg = [0.000319116, 0.000968288, 0.00320824, 0.00947269, 0.0311209, 0.0631523, 0.122319, 0.186175, 0.241094, 0.29566, 0.436553, 0.552997, 0.78373, 1.14108, 1.43167, 1.77024]

USER_PROMPT = (f"Here are pure-component CO2 and N2 isotherms at 298 K on a 13X-like adsorbent.\n"
               f"pressure (bar): {pressure_bar}\nCO2 loading (mol/kg): {co2_mol_per_kg}\n"
               f"N2 loading (mol/kg): {n2_mol_per_kg}\n\n"
               "Using IAST, what is the CO2/N2 selectivity for a flue gas with 15% CO2 at 1 bar total pressure?")

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]
first = model.invoke(messages)
show(first, "Attempt 1", "Read it critically: is there a number? How did it get it?")

Note what it gave you: a number, a range, or only a method.

Now send **exactly the same messages** again. The model doesn't remember
attempt 1: every call starts from nothing, because **the model is
stateless**. Chat apps only seem to remember because they resend the
whole conversation each time.

In [ ]:
second = model.invoke(messages)
show(second, "Attempt 2", "Same messages, fresh call, no memory of attempt 1. Does it agree?")

### What just happened

Running LASP's own Isotherm Fitter and IAST engines on this data gives
**S ≈ 388–420**: 388 if you fit N₂ with Langmuir, and up to 420 with the
other two N₂ models that fit equally well. You'll compute this in Lab 2.

Your answers will differ from run to run. Here's what the models said to
this exact question **in our test runs while preparing this workshop**
(October 2026):

| Model | Where | Answer |
|---|---|---|
| Gemini 3.5 Flash Lite | n8n | **25.6**, stated with confidence |
| Claude Haiku 5.5 | n8n | **≈ 115**, "a hand estimate" |
| Claude Haiku 5.5 | Colab, attempt 1 | **≈ 480**, "a hand-calculation estimate" |
| Claude Haiku 5.5 | Colab, attempt 2 | **≈ 390**, and it flagged the N₂ extrapolation |

Three things to take from that:

- **A wrong answer and a good one look equally confident.** 25.6 is off by
  a factor of 15, and nothing in its tone says so.
- **Even a careful answer isn't reproducible.** The same model gave 115,
  480 and 390.
- **Even when it's close, you can't check it.** Which fit did it use? How
  did it integrate the spreading pressure? You'd have to redo the work to
  trust the number.

## The gap, and what closes it

| A model can | A model can't |
|---|---|
| explain IAST, outline the method, reason about which model to use | run *your* solver, or give the same checked number twice |

**A tool closes the gap.** In Lab 2 the same model gets the LASP apps as
tools, through an MCP server, so the number comes from the solver.

### Your turn

Change only the system prompt. Can you make the model honest about what it
can't compute?

In [ ]:
STRICT_SYSTEM_PROMPT = "Never give a number you didn't compute with a tool. Say what you'd need instead."

show(model.invoke([
    {"role": "system", "content": STRICT_SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]), "With a stricter system prompt", "Same user prompt; only the system prompt changed.")

A system prompt can make the model **honest about the gap**. It can't
**close** it.

## If something breaks

| What you see | Fix |
|---|---|
| `404 NOT_FOUND`, model not available | The provider retired the model. Pick a current one (<https://aistudio.google.com> or <https://docs.anthropic.com/en/docs/about-claude/models>) and change it in `MODELS`. |
| `401`, `API key not valid`, `authentication_error` | Check the key's secret (`GOOGLE_API_KEY` or `ANTHROPIC_API_KEY`): exact name, Notebook access on, and that `PROVIDER` matches it. |
| `429`, `RESOURCE_EXHAUSTED` | Free-tier limit. Wait a minute and re-run the cell. |
| `ModuleNotFoundError`, or a name not defined | The runtime restarted. Run the setup cells again from the top. |